In [1]:
import os
import argparse
import subprocess
import numpy as np
import netCDF4 as nc
from scipy.interpolate import make_interp_spline
from tqdm.auto import tqdm


# ============================================================
# Exodus readers
# ============================================================
def unique_within_tolerance(arr, tol):
    arr = np.asarray(arr)
    sorted_arr = np.sort(arr)
    unique = [sorted_arr[0]]
    for i in range(1, len(sorted_arr)):
        if np.abs(sorted_arr[i] - unique[-1]) > tol:
            unique.append(sorted_arr[i])
    return np.array(unique)


def read_neu_to_np(file_path):
    dataset = nc.Dataset(file_path, "r")

    x_coords = dataset.variables["coordx"][:]
    y_coords = dataset.variables["coordy"][:]

    u = dataset.variables["vals_nod_var3"]

    unique_x = unique_within_tolerance(np.array(x_coords), 1e-6)
    unique_y = unique_within_tolerance(np.array(y_coords), 1e-3)

    time_steps = u.shape[0]
    z_matrix = np.zeros((time_steps, len(unique_x), len(unique_y)))
    mask = np.zeros((len(unique_x), len(unique_y)))

    for i in range(len(x_coords)):
        x_index = np.argmin(np.abs(x_coords[i] - unique_x))
        y_index = np.argmin(np.abs(y_coords[i] - unique_y))
        mask[x_index, y_index] = 1
        z_matrix[:, x_index, y_index] = u[:, i]

    # node -> cell-center style averaging
    z_matrix = (z_matrix[..., :-1] + z_matrix[..., 1:]) / 2
    z_matrix = (z_matrix[:, :-1] + z_matrix[:, 1:]) / 2

    assert np.mean(mask) == 1, "An element has not been assigned a value"
    dataset.close()
    return z_matrix


def read_fuel_to_np(file_path):
    dataset = nc.Dataset(file_path, "r")

    x_coords = dataset.variables["coordx"][:]
    y_coords = dataset.variables["coordy"][:]

    u = dataset.variables["vals_nod_var1"]
    flux = np.array(dataset.variables["vals_elem_var1eb1"]).reshape(1, u.shape[0], 64, 8).transpose(0, 1, 3, 2)

    unique_x = unique_within_tolerance(np.array(x_coords), 1e-6)
    unique_y = unique_within_tolerance(np.array(y_coords), 1e-3)

    time_steps = u.shape[0]
    z_matrix = np.zeros((1, time_steps, len(unique_x), len(unique_y)))
    mask = np.zeros((len(unique_x), len(unique_y)))

    for i in range(len(x_coords)):
        x_index = np.argmin(np.abs(x_coords[i] - unique_x))
        y_index = np.argmin(np.abs(y_coords[i] - unique_y))
        mask[x_index, y_index] = 1
        z_matrix[0, :, x_index, y_index] = u[:, i]

    # node -> cell-center style averaging
    z_matrix = (z_matrix[:, :, 1:, 1:] + z_matrix[:, :, :-1, :-1]) / 2

    assert np.mean(mask) == 1, "An element has not been assigned a value"
    dataset.close()
    return np.concatenate((z_matrix, flux), axis=0)


def read_fluid_to_np(file_path):
    dataset = nc.Dataset(file_path, "r")

    T_fluid = np.array(dataset.variables["vals_elem_var1eb1"]).reshape(1, -1, 64, 12)
    pressure = np.array(dataset.variables["vals_elem_var3eb1"]).reshape(1, -1, 64, 12)
    vel_x = np.array(dataset.variables["vals_elem_var4eb1"]).reshape(1, -1, 64, 12)
    vel_y = np.array(dataset.variables["vals_elem_var5eb1"]).reshape(1, -1, 64, 12)

    dataset.close()
    return np.concatenate((T_fluid, pressure, vel_x, vel_y), axis=0).transpose(0, 1, 3, 2)


# ============================================================
# phi BC generation
# ============================================================
def gen_phi_BC(*_args):
    def generate_x_coords(min_val, max_val, n_points, threshold):
        x_coords = []
        while len(x_coords) < n_points:
            x = np.random.uniform(min_val, max_val)
            if (
                (len(x_coords) == 0 or all(abs(x - xi) > threshold for xi in x_coords))
                and abs(x - min_val) > threshold
                and abs(x - max_val) > threshold
            ):
                x_coords.append(x)
        return sorted(x_coords)

    phi_all = []
    time_steps = 16

    while True:
        x_random = generate_x_coords(0, 0.75, 5, 0.075)
        y_random = np.random.uniform(0.5, 3, 5)
        max_index = np.argmax(y_random)

        x_fixed = np.array([0, 0.75])
        y_fixed = np.array([0.5, 0.5])

        x_all = np.concatenate(([x_fixed[0]], x_random, [x_fixed[1]]))
        y_all = np.concatenate(([y_fixed[0]], y_random, [y_fixed[1]]))

        spline = make_interp_spline(x_all, y_all)
        x_spline = np.linspace(0, 0.75, 65)
        y_spline = spline(x_spline)

        if np.all(y_spline >= 0):
            break

    phi_all.append(y_spline)

    max_increase = np.random.uniform(0.1, 0.9)
    for _t in range(1, time_steps):
        peak_increase = np.random.uniform(0.1, max_increase)
        factor = np.random.uniform(0.1, 0.7, 5)
        y_random += factor * peak_increase
        y_random[max_index] += (1 - factor[max_index]) * peak_increase

        y_all = np.concatenate(([y_fixed[0]], y_random, [y_fixed[1]]))
        spline = make_interp_spline(x_all, y_all)
        y_spline = np.abs(spline(x_spline))
        phi_all.append(np.abs(y_spline))

    phi_all = np.array(phi_all).transpose(1, 0)
    return phi_all


def replacements(
    function,
    tag="T",
    Lx=0.0076,
    Ly=0.75,
    Lt=5,
    nx=8,
    ny=64,
    nt=16,
    bias_x=0,
    bias_y=0,
    bias_t=0,
    dim=2,
):
    dx = Lx / nx
    dy = Ly / ny
    dt = Lt / nt

    coor_t_str = ""
    data = ""
    coor_x_str = "%.5f" % bias_x
    coor_y_str = "%.5f" % bias_y

    for i in range(nx):
        coor_x_str += " %.5f" % (dx * (i + 1) + bias_x)
    for i in range(ny):
        coor_y_str += " %.5f" % (dy * (i + 1) + bias_y)
    for i in range(nt):
        coor_t_str += "%.5f " % (dt * (i + 1) + bias_t)

    x_values = np.array(list(map(float, coor_x_str.split())))
    y_values = np.array(list(map(float, coor_y_str.split())))
    t_values = np.array(list(map(float, coor_t_str.split())))

    _X, _Y, _T = np.meshgrid(x_values, y_values, t_values, indexing="ij")
    Z = function()

    for i in range(nt):
        for j in range(len(y_values)):
            data += "%.2f " % Z[j, i]

    replacements_dict = {"y_coor": coor_y_str, "t_coor": coor_t_str, "data": data}
    return replacements_dict, Z


def write_inp(base_file, out_file, replacements_dict):
    with open(base_file, "r", encoding="utf-8") as base:
        template = base.read()
    src = template % replacements_dict
    with open(out_file, "w", encoding="utf-8") as file:
        file.write(src)


# ============================================================
# MOOSE runner
# ============================================================
def run_moose(input_file="solid.i", executable="../../workspace-opt", nproc=1):
    command = [executable, "-i", input_file]

    result = subprocess.run(command, capture_output=True, text=True)

    print("\n================ STDOUT ================\n")
    print(result.stdout if result.stdout else "(empty)")
    print("\n================ STDERR ================\n")
    print(result.stderr if result.stderr else "(empty)")
    print("\nReturn code:", result.returncode)

    if result.returncode != 0:
        raise RuntimeError("MOOSE run failed")

    return result


def verify_required_files():
    required = [
        "solid.i",
        "neutron.i",
        "fluid.i",
        "phi_base.txt",
    ]
    missing = [f for f in required if not os.path.exists(f)]
    if missing:
        raise FileNotFoundError(f"Missing required files: {missing}")

    # Optional check: only require local nft.e if neutron.i still uses a relative mesh path
    with open("neutron.i", "r", encoding="utf-8") as f:
        neutron_text = f.read()

    if "file = nft.e" in neutron_text and not os.path.exists("nft.e"):
        raise FileNotFoundError(
            "neutron.i still uses 'file = nft.e' but nft.e is not in the current directory. "
            "Either put nft.e here or change neutron.i to an absolute path."
        )


def verify_outputs():
    required_outputs = [
        "./solid_exodus.e",
        "./solid_out_sub_app0_exodus.e",
        "./solid_out_sub_app0_sub_app0_exodus.e",
    ]
    missing = [f for f in required_outputs if not os.path.exists(f)]
    if missing:
        raise FileNotFoundError(f"MOOSE finished but expected output files are missing: {missing}")


# ============================================================
# Data collection
# ============================================================
def collect_case_data():
    Tfuel = read_fuel_to_np("./solid_exodus.e")
    fluid = read_fluid_to_np("./solid_out_sub_app0_sub_app0_exodus.e")
    neutron = read_neu_to_np("./solid_out_sub_app0_exodus.e")
    return Tfuel, fluid, neutron


# ============================================================
# Main workflows
# ============================================================
def main(n=2):
    verify_required_files()
    os.makedirs("./output", exist_ok=True)

    phiBC_all = []
    T_fuel_all = []
    T_fluid_all = []
    neu_all = []

    for _i in tqdm(range(n), desc="calculate loop time step", total=n):
        replacements_phi, phiBC = replacements(
            function=gen_phi_BC,
            tag="phi",
            Lx=0.0076,
            Ly=0.75,
            Lt=5,
            nx=8,
            ny=64,
            nt=16,
            bias_x=0,
            bias_y=0,
            bias_t=0,
            dim=1,
        )

        write_inp("./phi_base.txt", "./phi.txt", replacements_phi)

        run_moose(input_file="solid.i", executable="../../workspace-opt", nproc=1)
        verify_outputs()

        Tfuel, fluid, neutron = collect_case_data()

        phiBC_all.append(phiBC)
        T_fuel_all.append(Tfuel)
        T_fluid_all.append(fluid)
        neu_all.append(neutron)

    np.save("./output/phiBC_to_phi.npy", np.array(phiBC_all))
    np.save("./output/nft_Tfuel.npy", np.array(T_fuel_all))
    np.save("./output/nft_Tfluid.npy", np.array(T_fluid_all))
    np.save("./output/nft_phi.npy", np.array(neu_all))

    print("Saved:")
    print("  ./output/phiBC_to_phi.npy")
    print("  ./output/nft_Tfuel.npy")
    print("  ./output/nft_Tfluid.npy")
    print("  ./output/nft_phi.npy")


def main1():
    verify_required_files()
    os.makedirs("./output", exist_ok=True)

    run_moose(input_file="solid.i", executable="../../workspace-opt", nproc=1)
    verify_outputs()

    Tfuel, fluid, neutron = collect_case_data()

    np.save("./output/nft_Tfuel.npy", np.array([Tfuel]))
    np.save("./output/nft_Tfluid.npy", np.array([fluid]))
    np.save("./output/nft_phi.npy", np.array([neutron]))

    print("Saved:")
    print("  ./output/nft_Tfuel.npy")
    print("  ./output/nft_Tfluid.npy")
    print("  ./output/nft_phi.npy")


# ============================================================
# Entry
# ============================================================
if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Generate data")
    parser.add_argument("--n", default=2000, type=int, help="number of samples")
    parser.add_argument("--type", default="single", type=str, help="single or batch")

    args, unknown = parser.parse_known_args()

    if args.type == "single":
        main1()
    else:
        main(args.n)

/env/miniconda3/envs/moose/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



================ STDOUT ================

(empty)

================ STDERR ================



*** ERROR ***
Unable to open file "/projects/549120ce-e7e0-45e0-a453-c9903649fea2/sqp/moose/nft.e-mesh.cpa.gz". Check to make sure that it exists and that you have read permission.

Stack frames: 23
0: libMesh::print_trace(std::ostream&)
1: moose::internal::mooseErrorRaw(std::__cxx11::basic_string<char, std::char_traits<char>, std::allocator<char> >, std::__cxx11::basic_string<char, std::char_traits<char>, std::allocator<char> > const&, hit::Node const*)
2: void mooseError<char const*>(char const*&&)
3: MooseUtils::checkFileReadable(std::__cxx11::basic_string<char, std::char_traits<char>, std::allocator<char> > const&, bool, bool, bool)
4: FileMesh::buildMesh()
5: MooseMesh::init()
6: SetupMeshAction::act()
7: Action::timedAct()
8: ActionWarehouse::executeActionsWithAction(std::__cxx11::basic_string<char, std::char_traits<char>, std::allocator<char> > const&)
9: ActionWarehouse::executeAllAc

RuntimeError: MOOSE run failed

In [8]:
!grep -R "use_split\|split_file\|restart_file_base\|recover\|checkpoint" solid.i neutron.i fluid.i

/usr/bin/mpiexec


mpiexec: Error: unknown option "--version!which"
Type 'mpiexec --help' for usage.


mpiexec (OpenRTE) 4.1.2

Report bugs to http://www.open-mpi.org/community/help/


returncode: 132
stdout:
 
stderr:
 --------------------------------------------------------------------------
mpiexec was unable to launch the specified application as it could not access
or execute an executable:

Executable: ../../workspace-opt
Node: 24aff760d95a

while attempting to start process rank 0.
--------------------------------------------------------------------------



../../cutest/cutest/include
../../cutest/cutest/src
../../cutest/sifdecode/doc/src
../../cutest/sifdecode/src
../../.vscode-server/extensions/ms-toolsai.vscode-jupyter-cell-tags-0.1.9/src
../../.vscode-server/extensions/ms-vscode.cpptools-1.30.5-linux-x64/dist/src
../../.vscode-server/extensions/ms-toolsai.vscode-jupyter-slideshow-0.1.6/src
../../.vscode-server/extensions/ms-vscode.cmake-tools-1.22.28/dist/src
../../.vscode-server/extensions/ms-vscode.cpp-devtools-0.3.6/dist/src
../../.cursor-server/extensions/ms-toolsai.vscode-jupyter-slideshow-0.1.6-universal/src
../../.cursor-server/extensions/ms-toolsai.vscode-jupyter-cell-tags-0.1.9-universal/src


In [11]:
import os
import numpy as np
from scipy.interpolate import make_interp_spline
import subprocess
import netCDF4 as nc
from tqdm.auto import tqdm
import argparse


def unique_within_tolerance(arr, tol):
    arr = np.asarray(arr, dtype=np.float64)
    sorted_arr = np.sort(arr)
    unique = [sorted_arr[0]]
    for i in range(1, len(sorted_arr)):
        if np.abs(sorted_arr[i] - unique[-1]) > tol:
            unique.append(sorted_arr[i])
    return np.array(unique, dtype=np.float64)


def read_neu_to_np(file_path):
    dataset = nc.Dataset(file_path, "r")

    x_coords = np.asarray(dataset.variables["coordx"][:], dtype=np.float64)
    y_coords = np.asarray(dataset.variables["coordy"][:], dtype=np.float64)
    u = np.asarray(dataset.variables["vals_nod_var3"][:], dtype=np.float64)

    unique_x = unique_within_tolerance(x_coords, 1e-6)
    unique_y = unique_within_tolerance(y_coords, 1e-3)

    time_steps = u.shape[0]
    z_matrix = np.zeros((time_steps, len(unique_x), len(unique_y)), dtype=np.float64)
    mask = np.zeros((len(unique_x), len(unique_y)), dtype=np.float64)

    for i in range(len(x_coords)):
        x_index = np.argmin(np.abs(x_coords[i] - unique_x))
        y_index = np.argmin(np.abs(y_coords[i] - unique_y))
        mask[x_index, y_index] = 1.0
        z_matrix[:, x_index, y_index] = u[:, i]

    z_matrix = (z_matrix[..., :-1] + z_matrix[..., 1:]) / 2.0
    z_matrix = (z_matrix[:, :-1] + z_matrix[:, 1:]) / 2.0

    assert np.mean(mask) == 1.0, "An element has not been assigned a value"

    dataset.close()
    return np.asarray(z_matrix, dtype=np.float64)


def read_fuel_to_np(file_path):
    dataset = nc.Dataset(file_path, "r")

    x_coords = np.asarray(dataset.variables["coordx"][:], dtype=np.float64)
    y_coords = np.asarray(dataset.variables["coordy"][:], dtype=np.float64)
    u = np.asarray(dataset.variables["vals_nod_var1"][:], dtype=np.float64)

    time_vals = np.asarray(dataset.variables["time_whole"][:], dtype=np.float64)
    num_time_steps = len(time_vals)

    flux = np.asarray(dataset.variables["vals_elem_var1eb1"][:], dtype=np.float64)
    flux = flux.reshape(1, num_time_steps, 64, 8).transpose(0, 1, 3, 2)

    unique_x = unique_within_tolerance(x_coords, 1e-6)
    unique_y = unique_within_tolerance(y_coords, 1e-3)

    z_matrix = np.zeros((1, u.shape[0], len(unique_x), len(unique_y)), dtype=np.float64)
    mask = np.zeros((len(unique_x), len(unique_y)), dtype=np.float64)

    for i in range(len(x_coords)):
        x_index = np.argmin(np.abs(x_coords[i] - unique_x))
        y_index = np.argmin(np.abs(y_coords[i] - unique_y))
        mask[x_index, y_index] = 1.0
        z_matrix[0, :, x_index, y_index] = u[:, i]

    z_matrix = (z_matrix[:, :, 1:, 1:] + z_matrix[:, :, :-1, :-1]) / 2.0

    assert np.mean(mask) == 1.0, "An element has not been assigned a value"

    out = np.concatenate((z_matrix, flux), axis=0).astype(np.float64)

    dataset.close()
    return out


def read_fluid_to_np(file_path):
    dataset = nc.Dataset(file_path, "r")

    time_vals = np.asarray(dataset.variables["time_whole"][:], dtype=np.float64)
    num_time_steps = len(time_vals)

    T_fluid = np.asarray(dataset.variables["vals_elem_var1eb1"][:], dtype=np.float64)
    pressure = np.asarray(dataset.variables["vals_elem_var3eb1"][:], dtype=np.float64)
    vel_x = np.asarray(dataset.variables["vals_elem_var4eb1"][:], dtype=np.float64)
    vel_y = np.asarray(dataset.variables["vals_elem_var5eb1"][:], dtype=np.float64)

    T_fluid = T_fluid.reshape(1, num_time_steps, 64, 12)
    pressure = pressure.reshape(1, num_time_steps, 64, 12)
    vel_x = vel_x.reshape(1, num_time_steps, 64, 12)
    vel_y = vel_y.reshape(1, num_time_steps, 64, 12)

    z_matrix = np.concatenate((T_fluid, pressure, vel_x, vel_y), axis=0).transpose(0, 1, 3, 2)

    dataset.close()
    return np.asarray(z_matrix, dtype=np.float64)


def gen_phi_BC(*arg):
    def generate_x_coords(min_val, max_val, n_points, threshold):
        x_coords = []
        while len(x_coords) < n_points:
            x = np.random.uniform(min_val, max_val)
            if (
                (len(x_coords) == 0 or all(abs(x - xi) > threshold for xi in x_coords))
                and abs(x - min_val) > threshold
                and abs(x - max_val) > threshold
            ):
                x_coords.append(x)
        return sorted(x_coords)

    phi_all = []
    time_steps = 16

    while True:
        x_random = generate_x_coords(0, 0.75, 5, 0.075)
        y_random = np.random.uniform(0.5, 3.0, 5)
        max_index = np.argmax(y_random)

        x_fixed = np.array([0.0, 0.75], dtype=np.float64)
        y_fixed = np.array([0.5, 0.5], dtype=np.float64)

        x_all = np.concatenate(([x_fixed[0]], x_random, [x_fixed[1]])).astype(np.float64)
        y_all = np.concatenate(([y_fixed[0]], y_random, [y_fixed[1]])).astype(np.float64)

        spline = make_interp_spline(x_all, y_all)
        x_spline = np.linspace(0.0, 0.75, 65, dtype=np.float64)
        y_spline = spline(x_spline)

        if np.all(y_spline >= 0):
            break

    phi_all.append(np.asarray(y_spline, dtype=np.float64))

    max_increase = np.random.uniform(0.1, 0.9)
    for _ in range(1, time_steps):
        peak_increase = np.random.uniform(0.1, max_increase)
        factor = np.random.uniform(0.1, 0.7, 5)
        y_random += factor * peak_increase
        y_random[max_index] += (1.0 - factor[max_index]) * peak_increase

        y_all = np.concatenate(([y_fixed[0]], y_random, [y_fixed[1]])).astype(np.float64)
        spline = make_interp_spline(x_all, y_all)
        y_spline = np.abs(spline(x_spline))
        phi_all.append(np.asarray(y_spline, dtype=np.float64))

    phi_all = np.array(phi_all, dtype=np.float64).transpose(1, 0)
    return phi_all


def replacements(function, tag="T", Lx=0.0076, Ly=0.75, Lt=5, nx=8, ny=64, nt=16,
                 bias_x=0, bias_y=0, bias_t=0, dim=2):
    dx = Lx / nx
    dy = Ly / ny
    dt = Lt / nt

    coor_t_str = ""
    data = ""
    coor_x_str = "%.5f" % bias_x
    coor_y_str = "%.5f" % bias_y

    for i in range(nx):
        coor_x_str += " %.5f" % (dx * (i + 1) + bias_x)

    for i in range(ny):
        coor_y_str += " %.5f" % (dy * (i + 1) + bias_y)

    for i in range(nt):
        coor_t_str += "%.5f " % (dt * (i + 1) + bias_t)

    x_values = np.array(list(map(float, coor_x_str.split())), dtype=np.float64)
    y_values = np.array(list(map(float, coor_y_str.split())), dtype=np.float64)
    t_values = np.array(list(map(float, coor_t_str.split())), dtype=np.float64)

    X, Y, T = np.meshgrid(x_values, y_values, t_values, indexing="ij")
    _ = (X, Y, T)

    Z = np.asarray(function(), dtype=np.float64)

    for i in range(nt):
        for j in range(len(y_values)):
            data += "%.2f " % Z[j, i]

    repl = {"y_coor": coor_y_str, "t_coor": coor_t_str, "data": data}
    return repl, Z


def write_inp(base_file, out_file, replacements_dict):
    with open(base_file, "r", encoding="utf-8") as base:
        template = base.read()
    src = template % replacements_dict
    with open(out_file, "w", encoding="utf-8") as file:
        file.write(src)


def run_moose():
    command = ["../../workspace-opt", "-i", "solid.i"]
    result = subprocess.run(command, text=True)
    print("Return code:", result.returncode)
    if result.returncode != 0:
        raise RuntimeError("MOOSE run failed")


def save_numeric_npy(path, arr):
    arr = np.asarray(arr, dtype=np.float64)
    print(f"saving {path}: shape={arr.shape}, dtype={arr.dtype}")
    np.save(path, arr)


def main(n=2):
    phiBC_all = []
    T_fuel_all = []
    T_fluid_all = []
    neu_all = []

    for i in tqdm(range(n), desc="calculate loop time step", total=n):
        print(f"\n========== sample {i+1}/{n} ==========")

        replacements_phi, phiBC = replacements(
            function=gen_phi_BC,
            tag="phi",
            Lx=0.0076,
            Ly=0.75,
            Lt=5,
            nx=8,
            ny=64,
            nt=16,
            bias_x=0,
            bias_y=0,
            bias_t=0,
            dim=1,
        )
        write_inp("./phi_base.txt", "./phi.txt", replacements_phi)

        run_moose()

        Tfuel = read_fuel_to_np("./solid_exodus.e")
        fluid = read_fluid_to_np("./solid_out_sub_app0_sub_app0_exodus.e")
        neutron = read_neu_to_np("./solid_out_sub_app0_exodus.e")

        print("phiBC   :", type(phiBC), phiBC.shape, phiBC.dtype)
        print("Tfuel   :", type(Tfuel), Tfuel.shape, Tfuel.dtype)
        print("Tfluid  :", type(fluid), fluid.shape, fluid.dtype)
        print("neutron :", type(neutron), neutron.shape, neutron.dtype)

        phiBC_all.append(np.asarray(phiBC, dtype=np.float64))
        T_fuel_all.append(np.asarray(Tfuel, dtype=np.float64))
        T_fluid_all.append(np.asarray(fluid, dtype=np.float64))
        neu_all.append(np.asarray(neutron, dtype=np.float64))

    phiBC_all = np.stack(phiBC_all, axis=0)
    T_fuel_all = np.stack(T_fuel_all, axis=0)
    T_fluid_all = np.stack(T_fluid_all, axis=0)
    neu_all = np.stack(neu_all, axis=0)

    print("\n========== final stacked arrays ==========")
    print("phiBC_all  :", phiBC_all.shape, phiBC_all.dtype)
    print("T_fuel_all :", T_fuel_all.shape, T_fuel_all.dtype)
    print("T_fluid_all:", T_fluid_all.shape, T_fluid_all.dtype)
    print("neu_all    :", neu_all.shape, neu_all.dtype)

    os.makedirs("./output", exist_ok=True)
    save_numeric_npy("./output/phiBC_to_phi.npy", phiBC_all)
    save_numeric_npy("./output/nft_Tfuel.npy", T_fuel_all)
    save_numeric_npy("./output/nft_Tfluid.npy", T_fluid_all)
    save_numeric_npy("./output/nft_phi.npy", neu_all)

    print("\nsaved all successfully")


def main1():
    phiBC_all = []
    T_fuel_all = []
    T_fluid_all = []
    neu_all = []

    run_moose()

    Tfuel = read_fuel_to_np("./solid_exodus.e")
    fluid = read_fluid_to_np("./solid_out_sub_app0_sub_app0_exodus.e")
    neutron = read_neu_to_np("./solid_out_sub_app0_exodus.e")

    print("Tfuel   :", type(Tfuel), Tfuel.shape, Tfuel.dtype)
    print("Tfluid  :", type(fluid), fluid.shape, fluid.dtype)
    print("neutron :", type(neutron), neutron.shape, neutron.dtype)

    T_fuel_all.append(np.asarray(Tfuel, dtype=np.float64))
    T_fluid_all.append(np.asarray(fluid, dtype=np.float64))
    neu_all.append(np.asarray(neutron, dtype=np.float64))

    T_fuel_all = np.stack(T_fuel_all, axis=0)
    T_fluid_all = np.stack(T_fluid_all, axis=0)
    neu_all = np.stack(neu_all, axis=0)

    print("\n========== final stacked arrays ==========")
    print("T_fuel_all :", T_fuel_all.shape, T_fuel_all.dtype)
    print("T_fluid_all:", T_fluid_all.shape, T_fluid_all.dtype)
    print("neu_all    :", neu_all.shape, neu_all.dtype)

    os.makedirs("./output", exist_ok=True)
    save_numeric_npy("./output/nft_Tfuel.npy", T_fuel_all)
    save_numeric_npy("./output/nft_Tfluid.npy", T_fluid_all)
    save_numeric_npy("./output/nft_phi.npy", neu_all)

    print("\nsaved all successfully")


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Generate data")
    parser.add_argument("--n", default=2000, type=int, help="number of sample")
    parser.add_argument("--type", default="single", type=str, help="single or batch")
    args, _ = parser.parse_known_args()

    if args.type == "single":
        main1()
    else:
        main(args.n)

ModuleNotFoundError: No module named 'netCDF4'

In [3]:
import numpy as np

a = np.load("./output/nft_Tfuel.npy")
b = np.load("./output/nft_Tfluid.npy")
c = np.load("./output/nft_phi.npy")

print(a.shape, a.dtype)
print(b.shape, b.dtype)
print(c.shape, c.dtype)

(1, 2, 17, 8, 64) float64
(1, 4, 17, 12, 64) float64
(1, 17, 20, 64) float64


In [0]:
"""u = vel_x
mu = aux_mu